# Notebook 05 — Agent-Based Modeling Fundamentals

**Module:** 15 — Simulation and Agent-Based Modeling  
**Tier:** 2 — Working competence  
**Notebook:** 5 of 12  
**Time estimate:** 75 minutes

> An agent-based model (ABM) simulates a system as a collection of autonomous
> individuals (agents), each following simple local rules. Global patterns emerge
> from agent interactions without being programmed explicitly. This notebook builds
> a minimal ABM framework from scratch, then implements Schelling segregation.

---
## Step 1 — Motivation

Thomas Schelling (1971) showed that racial segregation in US cities could emerge
from mild individual preferences — each person only wants to live near a few
similar neighbours (30–40% threshold), yet the result is nearly complete
segregation at the city level. No-one plans or desires full segregation — it
emerges from individual-level rules. This is the canonical example of ABM:
emergent macro-structure from micro-level rules, with no top-down design.

---
## Step 2 — Intuition

**ABM core components:**
1. **Agents:** autonomous individuals with state (position, color, age, wealth...)
   and a `step()` method that defines their behaviour each tick.
2. **Model:** the environment — contains the agent collection, a grid or network,
   and a random number generator.
3. **Scheduler:** determines the order in which agents step. Common types:
   - *Random activation:* agents step in a random order each tick
   - *Simultaneous activation:* all agents observe the current state, then
     all update at once (analogous to synchronous CA update)
   - *Stage activation:* separate movement and interaction stages
4. **Data collector:** records aggregate statistics (population counts, mean
   values) at each step — essential for analysis.

**Schelling model rules:**
- Grid of cells; each cell is empty, or occupied by type A or type B agent
- Satisfaction threshold τ: an agent is satisfied if ≥ τ fraction of its
  neighbours are the same type
- Unsatisfied agents move to a random empty cell
- Run until all agents are satisfied (or max steps)
- Measure: segregation index (fraction of neighbours same type, averaged)

---
## Step 3 — Biological Background

**ABM vs. ODE/PDE:**
ABMs capture heterogeneity (agents differ), discreteness (integer individuals),
and local interactions (no mean-field assumption). When are ABMs necessary?
- Small populations (stochasticity matters: N < 100 cells in a developing organ)
- Heterogeneous agents (different cell types, mutation load, drug resistance)
- Spatial structure (tissue architecture, biofilm formation)
- Emergent collective behaviour (cell sorting, swarming, flocking)

**Biological ABM examples:**
- *Cell sorting (Steinberg):* differential adhesion hypothesis — cells sort by
  adhesion strength, like oil and water. Implemented as ABM via Cellular Potts model.
- *Tumour growth:* ABM tracks individual cancer cells, their proliferation rate,
  oxygen consumption, drug resistance — impossible to capture in a PDE.
- *Immune system:* NetLogo's HIV model tracks individual CD4+ T cells and virions.
- *Biofilm formation:* bacteria consume nutrients locally, reproduce, and form
  spatial structure that feeds back on nutrient gradients.

**Mesa framework (Python):** `pip install mesa`. Provides Agent, Model,
MultiGrid, SingleGrid, DataCollector, and Scheduler classes. Used in
epidemiology, economics, and social simulation.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from collections import defaultdict

rng = np.random.default_rng(42)

# ============================================================
# Minimal ABM framework from scratch
# ============================================================

class Agent:
    """Base agent class."""
    def __init__(self, uid, model):
        self.uid = uid
        self.model = model
    def step(self):
        pass  # override in subclass

class Model:
    """Base model class with a random scheduler."""
    def __init__(self, seed=None):
        self.rng = np.random.default_rng(seed)
        self.agents = []
        self.schedule = []  # list of agent uids
        self.step_count = 0
        self.data = defaultdict(list)  # collected at each step

    def add_agent(self, agent):
        self.agents.append(agent)

    def run_step(self):
        """Random activation scheduler."""
        order = self.rng.permutation(len(self.agents))
        for i in order:
            self.agents[i].step()
        self.collect()
        self.step_count += 1

    def collect(self):
        pass  # override to record stats

    def run(self, n_steps):
        for _ in range(n_steps):
            self.run_step()

# ============================================================
# Schelling Segregation Model
# ============================================================

class SchellingAgent(Agent):
    def __init__(self, uid, model, pos, agent_type):
        super().__init__(uid, model)
        self.pos   = pos          # (row, col)
        self.atype = agent_type   # 0 or 1
        self.satisfied = False

    @property
    def satisfaction(self):
        """Fraction of occupied neighbours that share this agent's type."""
        r, c = self.pos
        N = self.model.N
        same = total = 0
        for dr in [-1, 0, 1]:
            for dc in [-1, 0, 1]:
                if dr == 0 and dc == 0: continue
                nr, nc = (r+dr) % N, (c+dc) % N
                occ = self.model.grid[nr, nc]
                if occ is not None:
                    total += 1
                    if occ.atype == self.atype:
                        same += 1
        return same / total if total > 0 else 1.0

    def step(self):
        frac_same = self.satisfaction
        if frac_same >= self.model.threshold:
            self.satisfied = True
        else:
            self.satisfied = False
            # Move to a random empty cell
            if self.model.empty_cells:
                new_pos = self.model.empty_cells[
                    self.model.rng.integers(0, len(self.model.empty_cells))]
                self.model.empty_cells.append(self.pos)
                self.model.grid[self.pos[0], self.pos[1]] = None
                self.pos = new_pos
                self.model.grid[new_pos[0], new_pos[1]] = self
                self.model.empty_cells.remove(new_pos)

class SchellingModel(Model):
    def __init__(self, N=40, density=0.80, threshold=0.4, seed=42):
        super().__init__(seed=seed)
        self.N = N
        self.threshold = threshold
        self.grid = np.full((N, N), None, dtype=object)  # None = empty
        self.empty_cells = []

        uid = 0
        positions = list(np.ndindex(N, N))
        self.rng.shuffle(positions)
        n_agents = int(density * N * N)
        for pos in positions[:n_agents]:
            atype = int(uid % 2)  # alternating types
            agent = SchellingAgent(uid, self, pos, atype)
            self.grid[pos[0], pos[1]] = agent
            self.add_agent(agent)
            uid += 1
        for pos in positions[n_agents:]:
            self.empty_cells.append(pos)

    def collect(self):
        # Fraction satisfied
        n_satisfied = sum(1 for a in self.agents if a.satisfied)
        self.data['satisfied'].append(n_satisfied / len(self.agents))
        # Segregation index (mean same-type neighbour fraction)
        seg = np.mean([a.satisfaction for a in self.agents])
        self.data['segregation'].append(seg)

    def grid_array(self, atype=None):
        """Return grid as integer array: -1=empty, 0=typeA, 1=typeB."""
        arr = np.full((self.N, self.N), -1, dtype=int)
        for r in range(self.N):
            for c in range(self.N):
                if self.grid[r, c] is not None:
                    arr[r, c] = self.grid[r, c].atype
        return arr

# Run model
print('Running Schelling model (threshold=0.4)...')
m40 = SchellingModel(threshold=0.4, seed=42)
grid_initial = m40.grid_array()
m40.run(80)
grid_final40 = m40.grid_array()
data40 = dict(m40.data)

print('Running Schelling model (threshold=0.6)...')
m60 = SchellingModel(threshold=0.6, seed=42)
m60.run(80)
grid_final60 = m60.grid_array()
data60 = dict(m60.data)

print(f'Threshold 0.4 — final segregation index: {data40["segregation"][-1]:.3f}')
print(f'Threshold 0.6 — final segregation index: {data60["segregation"][-1]:.3f}')

In [ ]:
# ---- Threshold sweep: segregation vs. tolerance ----
thresholds = np.arange(0.1, 0.9, 0.1)
final_segs = []
for tau in thresholds:
    m = SchellingModel(N=30, density=0.80, threshold=round(tau, 1), seed=7)
    m.run(100)
    final_segs.append(m.data['segregation'][-1])
print(f'Threshold sweep complete. Range: {min(final_segs):.3f} – {max(final_segs):.3f}')

In [ ]:
# Step 7 — Visualization
cmap_sch = mcolors.ListedColormap(['lightgrey', 'steelblue', 'tomato'])
norm_sch = mcolors.BoundaryNorm([-1.5, -0.5, 0.5, 1.5], cmap_sch.N)

fig, axes = plt.subplots(2, 3, figsize=(18, 11))

# Panel A: Initial grid
ax = axes[0, 0]
ax.imshow(grid_initial, cmap=cmap_sch, norm=norm_sch, interpolation='none')
ax.set_title('A. Initial state\n(random placement, threshold=0.4)')
ax.axis('off')

# Panel B: Final grid (threshold=0.4)
ax = axes[0, 1]
ax.imshow(grid_final40, cmap=cmap_sch, norm=norm_sch, interpolation='none')
ax.set_title(f'B. After 80 steps (threshold=0.4)\nSeg. index={data40["segregation"][-1]:.3f}')
ax.axis('off')

# Panel C: Final grid (threshold=0.6)
ax = axes[0, 2]
ax.imshow(grid_final60, cmap=cmap_sch, norm=norm_sch, interpolation='none')
ax.set_title(f'C. After 80 steps (threshold=0.6)\nSeg. index={data60["segregation"][-1]:.3f}')
ax.axis('off')

# Panel D: Satisfaction and segregation over time
ax = axes[1, 0]
steps = range(len(data40['satisfied']))
ax.plot(steps, data40['satisfied'],   'steelblue', lw=2, label='Satisfied (τ=0.4)')
ax.plot(steps, data60['satisfied'],   'steelblue', lw=2, ls='--', label='Satisfied (τ=0.6)')
ax_r = ax.twinx()
ax_r.plot(steps, data40['segregation'], 'tomato', lw=2, label='Segregation (τ=0.4)')
ax_r.plot(steps, data60['segregation'], 'tomato', lw=2, ls='--', label='Segregation (τ=0.6)')
ax.set_xlabel('Step'); ax.set_ylabel('Fraction satisfied', color='steelblue')
ax_r.set_ylabel('Segregation index', color='tomato')
ax.set_title('D. Satisfaction and segregation\nover model steps')
lines1, l1 = ax.get_legend_handles_labels()
lines2, l2 = ax_r.get_legend_handles_labels()
ax.legend(lines1+lines2, l1+l2, fontsize=7)

# Panel E: Segregation vs. tolerance threshold
ax = axes[1, 1]
ax.plot(thresholds, final_segs, 'o-', color='tomato', lw=2, ms=8)
ax.set_xlabel('Satisfaction threshold τ'); ax.set_ylabel('Final segregation index')
ax.set_title('E. Schelling result: segregation emerges\neven for mild tolerance thresholds')
ax.axhline(0.5, color='k', ls=':', lw=1, label='50% same-type')
ax.legend(fontsize=8)

# Panel F: ABM component diagram
ax = axes[1, 2]
ax.axis('off')
text = (
    'ABM Framework Components\n\n'
    'AGENT\n'
    '  • Unique ID\n'
    '  • State (pos, type, age, ...)\n'
    '  • step() method\n'
    '  • Reads environment\n'
    '  • Updates own state\n\n'
    'MODEL\n'
    '  • Agent collection\n'
    '  • Environment (grid / network)\n'
    '  • Shared RNG\n\n'
    'SCHEDULER\n'
    '  • Random: shuffle each step\n'
    '  • Simultaneous: all at once\n\n'
    'DATA COLLECTOR\n'
    '  • Records aggregates each step\n'
    '  • Used for time-series plots'
)
ax.text(0.05, 0.95, text, transform=ax.transAxes, fontsize=9,
          va='top', fontfamily='monospace',
          bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.8))
ax.set_title('F. ABM architecture')

plt.suptitle('Module 15 NB05: Agent-Based Modeling Fundamentals', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('abm_fundamentals.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Step 8 — Exercises

1. Add a data collector that tracks the number of unsatisfied agents at each step.
   Plot this alongside the segregation index. Does full satisfaction always coincide
   with maximum segregation?
2. Implement the *simultaneous* activation scheduler (all agents observe state,
   then all update at once). Does it change the equilibrium segregation level?
3. Add a third agent type to the Schelling model. How does the model's behaviour
   change? Does it still converge to high segregation?

---
## Step 10 — Quiz

1. What is emergent behaviour in an ABM? Give one example from the Schelling model
   and one from biology.
2. What is the difference between random activation and simultaneous activation?
   Which is more realistic for gene expression networks and why?
3. Why does the Schelling model produce full segregation even with a threshold of
   only 30%? (Explain the cascade mechanism.)
4. Name three situations where an ABM is preferable to an ODE model for
   simulating a biological system.

---
## Step 12 — Reflection

> *[In one paragraph: the Schelling model is often cited as showing that segregation
> can emerge without racist intent — just mild preference for some similar neighbours.
> A critic argues this trivializes structural racism. How would you respond, and
> what does this debate tell you about the limits of ABM as a scientific tool?]*

---
*Next: `06_population_dynamics_and_evolution.ipynb`*